# EIGSEP Signal Recovery (v001)

- Composed objects: `Observer`, `Beam`, `Sky`, `Terrain`, `ForwardModel`
- Spectral basis decomposition: `BeamBasis`, `SkyBasis`
- Joint optimization: `Calibrator` with Anderson Acceleration + JAX autodiff

In [ ]:
import os
import numpy as np
import healpy
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u

from eigsep_sim import (
    EarthSurface, Beam, Sky, NullTerrain, ForwardModel, Calibrator,
    DTYPE_R_NPY, DTYPE_R_JAX
)
from eigsep_sim.recovery import RecoverySolution, ScaleDegeneracy
from eigsep_sim.models import T21cmModel
from eigsep_sim.linear_solver import normal_solve
from eigsep_sim.spectral import gsm_eigenmodes, eigenmode_filter

# Configuration
NSIDE = 8
N_FREQ = 20
NPIX = healpy.nside2npix(NSIDE)
LAT_DEG, LON_DEG = 39.2, -113.4
FREQS_MHZ = np.linspace(55.0, 150.0, N_FREQ)
FREQS_HZ = FREQS_MHZ * 1e6
DELTA_NU_HZ = float(np.diff(FREQS_MHZ).mean()) * 1e6
OBS_EPOCH = Time("2025-01-01")
N_TIMES = 36
N_AZ, N_ALT = 6, 5
N_ORIENT = N_AZ * N_ALT
N_BEAM_POLS = 1
if N_BEAM_POLS not in (1, 2):
    raise ValueError("N_BEAM_POLS must be 1 or 2")
BEAM_AXES_BODY = np.eye(3, dtype=DTYPE_R_NPY)[:N_BEAM_POLS]
AZ_RAD = np.linspace(0.0, 2.0 * np.pi, N_AZ, endpoint=False)
ALT_RAD = np.linspace(0.0, 0.5 * np.pi, N_ALT)
ORIENT_ROTS = np.stack([
    Beam.top2body(az, alt) for alt in ALT_RAD for az in AZ_RAD
])
T_RX_K = 100.0

print(f"Config: NSIDE={NSIDE}, N_FREQ={N_FREQ}, N_TIMES={N_TIMES}, "
      f"N_ORIENT={N_ORIENT}, N_BEAM_POLS={N_BEAM_POLS}")

In [ ]:
# Create Observer, Beam, Sky
obs = EarthSurface(lat=LAT_DEG, lon=LON_DEG)
beam = Beam.from_dipole(nside=8, freqs_hz=FREQS_HZ, arm_lengths_m=2.0, u_body=BEAM_AXES_BODY, K=5)
sky = Sky.from_gsm(NSIDE, FREQS_HZ, n_modes=5, include_flat=True)
gsm_maps = sky.init_coeffs()
gsm_maps_recon = gsm_maps @ sky.basis.A.T
fwd = ForwardModel(obs, beam, sky, terrain=NullTerrain())

In [ ]:
# Precompute geometry for every sidereal time and beam orientation
base_times = OBS_EPOCH + np.linspace(0, 86400, N_TIMES, endpoint=False) * u.s
base_rots = obs.rot_gal2top_stack(base_times)
rots = np.repeat(base_rots, N_ORIENT, axis=0)
body_rots = np.tile(ORIENT_ROTS, (N_TIMES, 1, 1))
geom = fwd.precompute_geometry(rots=rots, body_rots=body_rots)

# Forward simulation
sky_coeffs = gsm_maps
beam_coeffs = beam.coeffs.copy()
antenna_temp = fwd.simulate(sky_coeffs, beam_coeffs, geom=geom)

In [ ]:
# Create synthetic observations with noise
tau_per_obs = 86400.0 / (N_TIMES * N_ORIENT)

def sampled_beam_weights(geom, beam_coeffs, beam_basis_A):
    """Return per-sample integrated beam weights in simulator units."""
    beam_maps = beam_coeffs @ beam_basis_A.T
    pixels = np.asarray(geom['beam_px_jax'])
    weights = np.asarray(geom['beam_wgts_jax'])
    ntimes = pixels.shape[0]
    n_dipoles, _, nfreq = beam_maps.shape
    beam_weights = np.zeros((ntimes, n_dipoles, nfreq), dtype=float)
    for freq_index in range(nfreq):
        for dipole_index in range(n_dipoles):
            beam_map = beam_maps[dipole_index, :, freq_index]
            beam_weights[:, dipole_index, freq_index] = sum(
                (beam_map[pixels[:, neighbor_index, :]]
                 * weights[:, neighbor_index, :]).sum(axis=1)
                for neighbor_index in range(4)
            )
    return beam_weights

beam_weight = sampled_beam_weights(geom, beam_coeffs, beam.basis.A)
sigma_noise = (
    np.abs(np.asarray(antenna_temp))
    + T_RX_K * np.abs(beam_weight)
) / np.sqrt(DELTA_NU_HZ * tau_per_obs)

antenna_temp_flat = np.asarray(antenna_temp).reshape(-1, N_FREQ)
sigma_noise_flat = sigma_noise.reshape(-1, N_FREQ)
rng = np.random.default_rng(seed=42)
noise = rng.normal(scale=sigma_noise_flat, size=antenna_temp_flat.shape)
data_noisy = antenna_temp_flat + noise
inv_noise_var = 1.0 / sigma_noise_flat**2

print(f"  σ_noise: {sigma_noise.mean():.3f} simulator units")


In [ ]:
# Initialize and run Calibrator
cal = Calibrator(
    fwd=fwd,
    data=data_noisy,
    inv_noise_var=inv_noise_var,
    m_anderson=5,
    lam_beam=0.01,
    lam_sky=0.0
)

# Initialize from GSM sky (realistic starting point) with small perturbations
# to demonstrate joint sky+beam recovery
prms_tru = {'sky_coeffs': gsm_maps, 'beam_coeffs': beam_coeffs}
prms_ini = {k: v * np.random.uniform(0.9, 1.1, size=v.shape) for k, v in prms_tru.items()}
prms_opt = {k: v.copy() for k, v in prms_ini.items()}

In [ ]:
print(f"Running fit...", flush=True)
result = cal.fit(
    params=prms_opt, geom=geom, verbose=True,
    solver='adaptive-scheduled',
    # Optional final-loss polish; slower in delta-chi2/time.
    # schedule_lbfgs_max_every=10,
    # schedule_lbfgs_min_iter=20,
    # schedule_lbfgs_maxiter=3,
    # schedule_lbfgs_max_runs=1,
    max_iter=30, tol=1e-4,
    lambda_damp=1e-1,

)
prms_opt = result['params']
print(f"Converged: {result['converged']}")

In [ ]:
CH = 10
sky_ini = prms_ini['sky_coeffs'] @ sky.basis.A.T
sky_opt = prms_opt['sky_coeffs'] @ sky.basis.A.T
beam_ini = prms_ini['beam_coeffs'] @ beam.basis.A.T
beam_opt = prms_opt['beam_coeffs'] @ beam.basis.A.T
beam_recon = beam.coeffs @ beam.basis.A.T
scale_degens = [ScaleDegeneracy({'sky': 1.0, 'beam': -1.0}, group_axes=(-1,))]
map_ref = {'sky': gsm_maps_recon, 'beam': beam_recon}
maps_ini = RecoverySolution({'sky': sky_ini, 'beam': beam_ini}, scale_degens).remove_degen(map_ref, inplace=False).params
maps_opt = RecoverySolution({'sky': sky_opt, 'beam': beam_opt}, scale_degens).remove_degen(map_ref, inplace=False).params
sky_ini, beam_ini = maps_ini['sky'], maps_ini['beam']
sky_opt, beam_opt = maps_opt['sky'], maps_opt['beam']
healpy.mollview(                        sky_ini[:, CH], sub=(2, 2, 1), cmap='plasma', title='Initial')
healpy.mollview(sky_ini[:, CH] - gsm_maps_recon[:, CH], sub=(2, 2, 2), cmap='bwr'   , title='Diff')
healpy.mollview(                        sky_opt[:, CH], sub=(2, 2, 3), cmap='plasma', title='Solved')
healpy.mollview(sky_opt[:, CH] - gsm_maps_recon[:, CH], sub=(2, 2, 4), cmap='bwr'   , title='Diff')

In [ ]:
CH = 10
healpy.mollview(                      beam_ini[0, :, CH], sub=(2, 2, 1), cmap='plasma', title='Initial')
healpy.mollview(beam_ini[0,:, CH] - beam_recon[0, :, CH], sub=(2, 2, 2), cmap='bwr'   , title='Diff')
healpy.mollview(                      beam_opt[0, :, CH], sub=(2, 2, 3), cmap='plasma', title='Solved')
healpy.mollview(beam_opt[0,:, CH] - beam_recon[0, :, CH], sub=(2, 2, 4), cmap='bwr'   , title='Diff')

In [ ]:
import pandas as pd
pd.DataFrame(result["telemetry"])[[
    "loss", "step_type",
    "joint_step", "sky_step", "beam_step",
    "joint_loss", "sky_loss", "beam_loss",
    "sky_update_rms", "beam_update_rms",
    "beam_roughness",
]]